In [1]:
from pathlib import Path
import pandas as pd

# Locate the raw dataset from either the repository root or notebooks folder.
possible_paths = [
    Path("data/raw/bank-full.csv"),
    Path("../data/raw/bank-full.csv")
]

DATA_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError("Could not locate data/raw/bank-full.csv")

# Load the semicolon-separated UCI Bank Marketing dataset.
df = pd.read_csv(DATA_PATH, sep=";")

print(f"Dataset location: {DATA_PATH.resolve()}")
print(f"Dataset dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")

print("\nColumn data types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(df.isna().sum().to_frame("Missing Values"))

print(f"\nDuplicate records: {df.duplicated().sum():,}")

print("\nTarget distribution:")
display(df["y"].value_counts().to_frame("Count"))

print("\nFirst five records:")
display(df.head())

Dataset location: C:\Users\benja\OneDrive\Desktop\Lecture Notes\Degree BCSCUN\Sem 9\6001CMD Machine Learning\CW2 Project\6001CMD-ML-CW2\data\raw\bank-full.csv
Dataset dimensions: 45,211 rows × 17 columns

Column data types:


,Data Type
age,int64
job,object
marital,object
education,object
default,object
balance,int64
housing,object
loan,object
contact,object
day,int64



Missing values:


,Missing Values
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day,0



Duplicate records: 0

Target distribution:


,Count
y,
no,39922
yes,5289



First five records:


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [2]:
# Numerical feature summaries
print("NUMERICAL FEATURE SUMMARY")
numeric_summary = df.select_dtypes(include="number").describe().T
display(numeric_summary.round(2))

# Count explicit 'unknown' values in categorical predictors
print("\nUNKNOWN VALUES IN CATEGORICAL FEATURES")

categorical_cols = [
    col for col in df.select_dtypes(include="object").columns
    if col != "y"
]

unknown_summary = pd.DataFrame({
    "Unknown Count": [
        df[col].eq("unknown").sum() for col in categorical_cols
    ],
    "Unknown Percentage (%)": [
        df[col].eq("unknown").mean() * 100 for col in categorical_cols
    ]
}, index=categorical_cols)

display(unknown_summary.round(2))

# Examine special values in previous campaign information
print("\nPREVIOUS CAMPAIGN INFORMATION")

print(
    f"Customers never contacted in a previous campaign (pdays = -1): "
    f"{(df['pdays'] == -1).sum():,}"
)

print(
    f"Customers with no previous campaign contacts (previous = 0): "
    f"{(df['previous'] == 0).sum():,}"
)

NUMERICAL FEATURE SUMMARY


,count,mean,std,min,25%,50%,75%,max
age,45211.0,40.94,10.62,18.0,33.0,39.0,48.0,95.0
balance,45211.0,1362.27,3044.77,-8019.0,72.0,448.0,1428.0,102127.0
day,45211.0,15.81,8.32,1.0,8.0,16.0,21.0,31.0
duration,45211.0,258.16,257.53,0.0,103.0,180.0,319.0,4918.0
campaign,45211.0,2.76,3.10,1.0,1.0,2.0,3.0,63.0
pdays,45211.0,40.20,100.13,-1.0,-1.0,-1.0,-1.0,871.0
previous,45211.0,0.58,2.30,0.0,0.0,0.0,0.0,275.0



UNKNOWN VALUES IN CATEGORICAL FEATURES


,Unknown Count,Unknown Percentage (%)
job,288,0.64
marital,0,0.00
education,1857,4.11
default,0,0.00
housing,0,0.00
loan,0,0.00
contact,13020,28.80
month,0,0.00
poutcome,36959,81.75



PREVIOUS CAMPAIGN INFORMATION
Customers never contacted in a previous campaign (pdays = -1): 36,954
Customers with no previous campaign contacts (previous = 0): 36,954


In [3]:
# Check relationships between previous campaign indicators

pdays_missing = df["pdays"].eq(-1)
no_previous_contacts = df["previous"].eq(0)
unknown_outcome = df["poutcome"].eq("unknown")

print("PDAYS vs PREVIOUS")
display(pd.crosstab(
    pdays_missing,
    no_previous_contacts,
    rownames=["pdays = -1"],
    colnames=["previous = 0"],
    margins=True
))

print("\nPDAYS vs PREVIOUS CAMPAIGN OUTCOME")
display(pd.crosstab(
    pdays_missing,
    unknown_outcome,
    rownames=["pdays = -1"],
    colnames=["poutcome = unknown"],
    margins=True
))

PDAYS vs PREVIOUS


previous = 0,False,True,All
pdays = -1,,,
False,8257,0,8257
True,0,36954,36954
All,8257,36954,45211



PDAYS vs PREVIOUS CAMPAIGN OUTCOME


poutcome = unknown,False,True,All
pdays = -1,,,
False,8252,5,8257
True,0,36954,36954
All,8252,36959,45211


In [4]:
# Inspect customers with previous campaign contacts but unknown outcomes

exceptions = df.loc[
    (df["pdays"] >= 0) & (df["poutcome"] == "unknown")
]

display(exceptions)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
40658,61,retired,married,tertiary,no,3140,yes,yes,cellular,6,aug,975,4,98,1,unknown,yes
41821,39,management,married,tertiary,no,184,no,no,cellular,15,oct,206,1,168,5,unknown,no
42042,26,admin.,single,secondary,no,338,no,no,cellular,29,oct,209,1,188,2,unknown,yes
43978,30,technician,single,secondary,no,254,yes,yes,cellular,24,jun,167,3,416,2,unknown,no
45021,37,management,married,secondary,no,209,no,no,cellular,14,oct,183,3,528,7,unknown,no
